In [1]:
import pandas as pd
import duckdb as db
import requests
from io import StringIO

In [4]:
tpa = db.read_csv("../../data/TrafficPerAirport.csv")
tpt = db.read_csv("../../data/TrafficPerTerritory.csv")
territory = db.read_csv('../../data/Territory.csv')
airservice = db.read_csv('../../data/AirService.csv')
aircraftmovement = db.read_csv('../../data/AircraftMovement.csv')
airport = db.read_csv('../../data/Airport.csv')

In [ ]:
"""
I will be using this join a lot:
db.sql('SELECT * \
        FROM tpt INNER JOIN territory T ON T.TerritoryId = tpt.IslandId \
        INNER JOIN territory T2 ON T2.TerritoryId = tpt.StopoverTerritoryId')
"""

## Total observed values per Island in the last N months

[Date functions in duckdb](https://duckdb.org/docs/stable/sql/functions/date.html)

DATE_SUB vs DATE_DIFF

In [ ]:
# Last year
Nmonths = 12
# 0 arrival, 1 departure
aircraftsMovementsAllowed = [0, 1]

In [ ]:
listbetparenthesis = "("
for a in aircraftsMovementsAllowed: 
    listbetparenthesis += f"{a},"
listbetparenthesis = listbetparenthesis[0:-1] # Delete last comma
listbetparenthesis += ")"

In [ ]:
listbetparenthesis

'(0,1)'

In [101]:
if Nmonths == "total":
    Nmonths = db.sql("SELECT DATE_SUB('month', MIN(Month), MAX(Month)) FROM tpt").fetchone()[0]

In [102]:

db.sql( f"\
        SELECT T.TerritoryName, SUM(Passengers) AS Total_Passengers, SUM(Operations) AS Total_Op, SUM(Goods) AS Total_Goods, SUM(Mail) AS Total_Mail \
        FROM tpt INNER JOIN territory T ON T.TerritoryId = tpt.IslandId \
        WHERE tpt.AircraftMovementId IN {listbetparenthesis} AND T.TerritoryName != 'Canary Islands' \
        AND DATE_SUB('month', Month, (SELECT MAX(month) FROM tpt)) <= {Nmonths}  \
        GROUP BY T.TerritoryName \
        ORDER BY Total_Passengers DESC")

┌───────────────┬──────────────────┬──────────┬─────────────┬────────────┐
│ TerritoryName │ Total_Passengers │ Total_Op │ Total_Goods │ Total_Mail │
│    varchar    │      int128      │  int128  │   int128    │   int128   │
├───────────────┼──────────────────┼──────────┼─────────────┼────────────┤
│ Tenerife      │         61069064 │   454848 │    21168118 │    7191922 │
│ Gran Canaria  │         42089972 │   350968 │    41297158 │    3065838 │
│ Lanzarote     │         27773260 │   205714 │     1099578 │       1064 │
│ Fuerteventura │         21586922 │   157154 │      785904 │          0 │
│ La Palma      │          3645070 │    52818 │      632312 │         32 │
│ El Hierro     │           698782 │    13390 │      130458 │          8 │
│ La Gomera     │           274194 │     6292 │        7878 │          8 │
└───────────────┴──────────────────┴──────────┴─────────────┴────────────┘

## Total observed values in TrafficPerAirport filtered by BaseAirport, StopoverAirport, airservice, aricraftmovement  in the between N and M dates

In [125]:
s_date = '2024-01-01'
end_date = '2025-07-01'

In [121]:
# 0 arrival, 1 departure
aircraftmov = [0, 1]
# 0 Commercial (total), 1 Other, 2 Non scheduled, 3 regular
airservice = [0]

In [122]:
def list_to_string(list): 
    st = "("
    for a in list: 
        st += f"{a},"
    st = st[0:-1] # Delete last comma
    st += ")"
    return st

In [123]:
aircraftmovstr = list_to_string(aircraftmov)

airservicestr = list_to_string(airservice)

In [ ]:


db.sql( f"\
        SELECT BaseAirportId, StopoverAirportId, SUM(Passengers) AS Total_Passengers, SUM(Operations) AS Total_Op, SUM(Goods) AS Total_Goods, SUM(Mail) AS Total_Mail \
        FROM tpa \
        WHERE AircraftMovementId IN {aircraftmovstr} AND AirServiceId IN {airservicestr}\
        AND Month BETWEEN '{s_date}' AND '{end_date
                                           }'  \
        GROUP BY BaseAirportId, StopoverAirportId \
        ORDER BY Total_Passengers DESC")

┌───────────────┬───────────────────┬──────────────────┬──────────┬─────────────┬────────────┐
│ BaseAirportId │ StopoverAirportId │ Total_Passengers │ Total_Op │ Total_Goods │ Total_Mail │
│     int64     │       int64       │      int128      │  int128  │   int128    │   int128   │
├───────────────┼───────────────────┼──────────────────┼──────────┼─────────────┼────────────┤
│            47 │               111 │          3091032 │    20857 │    19502044 │     576386 │
│           137 │               111 │          2815571 │    18819 │     8369751 │    3222134 │
│            47 │               137 │          1562642 │    28262 │     2999526 │    1626394 │
│           137 │                47 │          1559923 │    28270 │     5145590 │    2104291 │
│           165 │                43 │          1524349 │     7780 │       46991 │          0 │
│            47 │                54 │          1346820 │    24331 │      362237 │        242 │
│            54 │                47 │          134

## Total observed values in TrafficPerAirport filtered by stopover airport country  in the between N and M dates